<a href="https://colab.research.google.com/github/HugoCrainich/Statistical-Machine-Learning/blob/main/Lab%206/Lab_6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [18]:
import seaborn as sns
import pandas as pd
import numpy as np

# 1. Data Ingestion (The Population)
df = sns.load_dataset('titanic')
print(f"Total Population: {len(df)}")
print(f"Population Survival Rate: {df['survived'].mean():.4f}")

# 2. Manual Shuffle (Simulation of Sampling)
# We set a seed to ensure reproducibility for the lesson,
# but in production, this variance happens naturally.
np.random.seed(2026)
indices = np.random.permutation(len(df))

Total Population: 891
Population Survival Rate: 0.3838


In [24]:
split_point = int(0.8 * len(df))

# Slicing the shuffled indices
train_idx = indices[:split_point]
test_idx = indices[split_point:]

# Creating the subsets
train_set = df.iloc[train_idx]
test_set = df.iloc[test_idx]

# 4. Bias Check (The Delta)
train_surv = train_set['survived'].mean()
test_surv = test_set['survived'].mean()
delta = abs(train_surv - test_surv)

print(f"Train Survival Rate: {train_surv:.4f}")
print(f"Test Survival Rate:  {test_surv:.4f}")
print(f"Sampling Bias (Delta): {delta:.4f}")


Train Survival Rate: 0.3736
Test Survival Rate:  0.4246
Sampling Bias (Delta): 0.0510


In [25]:
# WITH AI
from sklearn.model_selection import train_test_split

# Stratify by 'pclass' ensures the distribution of classes is identical
X_train, X_test = train_test_split(
    df,
    test_size=0.2,
    stratify=df['pclass'],  # This is the key: force identical class distributions
    random_state=2026
)

print("\n--- Stratified Split ---")
print("Train Class Dist:\n", X_train['pclass'].value_counts(normalize=True).sort_index())
print("\nTest Class Dist:\n", X_test['pclass'].value_counts(normalize=True).sort_index())

# Bonus: Check if stratification reduced survival rate bias
train_surv_strat = X_train['survived'].mean()
test_surv_strat = X_test['survived'].mean()
delta_strat = abs(train_surv_strat - test_surv_strat)

print(f"\n--- Bias Comparison ---")
print(f"Random Split Delta: {delta:.4f}")
print(f"Stratified Split Delta: {delta_strat:.4f}")
print(f"Improvement: {((delta - delta_strat) / delta * 100):.1f}% reduction in bias")


--- Stratified Split ---
Train Class Dist:
 pclass
1    0.242978
2    0.206461
3    0.550562
Name: proportion, dtype: float64

Test Class Dist:
 pclass
1    0.240223
2    0.206704
3    0.553073
Name: proportion, dtype: float64

--- Bias Comparison ---
Random Split Delta: 0.0510
Stratified Split Delta: 0.0720
Improvement: -41.1% reduction in bias


In [26]:
# WITH AI
from scipy.stats import chisquare

# 1. Define observed and expected arrays
observed = [450, 550]  # Control, Treatment
expected = [500, 500]  # Expected 50/50 split

# 2. Calculate Chi-Square statistic and p-value
chi2_stat, p_value = chisquare(f_obs=observed, f_exp=expected)

print(f"Chi-Square Statistic: {chi2_stat:.4f}")
print(f"P-value: {p_value:.6f}")
print()

# 3. Print conclusion
if p_value < 0.01:
    print("CRITICAL FAILURE: Sample Ratio Mismatch (SRM) Detected. Check Load Balancer.")
else:
    print("Variance is within natural limits.")

Chi-Square Statistic: 10.0000
P-value: 0.001565

CRITICAL FAILURE: Sample Ratio Mismatch (SRM) Detected. Check Load Balancer.
